<a href="https://colab.research.google.com/github/Dashami1310/AURUM-AI-/blob/main/Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchaudio transformers datasets accelerate \
             librosa soundfile peft bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.0 MB/s eta 0:00:00


In [ ]:
import torch
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from transformers import (
    WhisperProcessor,
    WhisperModel,
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset, Audio
from torch import nn
from torch.utils.data import DataLoader
import warnings
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
class AudioEncoder(nn.Module):
    def __init__(self, model_name="openai/whisper-small"):
        super().__init__()
        self.processor = WhisperProcessor.from_pretrained(model_name)
        self.encoder   = WhisperModel.from_pretrained(model_name).encoder.to(device)  # ADD .to(device)
        self.hidden_dim = self.encoder.config.d_model

        for param in self.encoder.parameters():
            param.requires_grad = False

    def forward(self, audio_array, sampling_rate=16000):
        if sampling_rate != 16000:
            resampler = torchaudio.transforms.Resample(sampling_rate, 16000)
            audio_array = resampler(torch.tensor(audio_array).float())

        inputs = self.processor(
            audio_array,
            sampling_rate=16000,
            return_tensors="pt"
        ).input_features.to(device)  # already there, good

        with torch.no_grad():
            encoder_output = self.encoder(inputs)

        return encoder_output.last_hidden_state

In [ ]:
class AudioToLLMAdapter(nn.Module):
    def __init__(self, audio_dim=512, llm_dim=2048, num_query_tokens=32):
        super().__init__()
        self.num_query_tokens = num_query_tokens

        self.query_tokens = nn.Parameter(
            torch.randn(1, num_query_tokens, audio_dim)
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=audio_dim,
            num_heads=8,
            batch_first=True
        )

        self.projection = nn.Sequential(
            nn.Linear(audio_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim)
        )

        self.layer_norm = nn.LayerNorm(llm_dim)

    def forward(self, audio_features):
        batch_size = audio_features.shape[0]
        queries = self.query_tokens.expand(batch_size, -1, -1)

        attended, _ = self.cross_attention(
            query=queries,
            key=audio_features,
            value=audio_features
        )

        projected = self.projection(attended)
        return self.layer_norm(projected)

In [ ]:
class AudioLanguageModel(nn.Module):
    def __init__(
        self,
        whisper_model="openai/whisper-small",
        llm_model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    ):
        super().__init__()

        # --- LISTEN ---
        self.audio_encoder = AudioEncoder(whisper_model)
        audio_dim = self.audio_encoder.hidden_dim

        # --- LLM setup ---
        print("Loading LLM tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(llm_model)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        print("Loading LLM model (this takes ~1 min)...")
        self.llm = AutoModelForCausalLM.from_pretrained(
            llm_model,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            device_map="auto"
        )
        llm_dim = self.llm.config.hidden_size

        for param in self.llm.parameters():
            param.requires_grad = False

        # --- THINK --- cast adapter to same dtype as LLM immediately
        model_dtype = torch.float16 if device == "cuda" else torch.float32
        self.adapter = AudioToLLMAdapter(
            audio_dim=audio_dim,
            llm_dim=llm_dim,
            num_query_tokens=32
        ).to(device=device, dtype=model_dtype)  # <-- key fix

    def encode_audio(self, audio_array, sampling_rate=16000):
        audio_features = self.audio_encoder(audio_array, sampling_rate)
        # cast audio features to match adapter dtype before passing in
        audio_features = audio_features.to(dtype=next(self.adapter.parameters()).dtype)
        audio_tokens = self.adapter(audio_features)
        return audio_tokens

    def generate_response(self, audio_array, prompt="Describe what you hear:", max_new_tokens=200):
        self.eval()

        # 1. Encode audio
        audio_tokens = self.encode_audio(audio_array)

        # 2. Encode text prompt
        text_inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            padding=True
        ).to(device)

        text_embeds = self.llm.get_input_embeddings()(text_inputs.input_ids)

        # 3. Both already float16 — safe to concat
        combined = torch.cat([audio_tokens, text_embeds], dim=1)

        # 4. Attention mask for full sequence
        seq_len = combined.shape[1]
        attention_mask = torch.ones(1, seq_len, dtype=torch.long, device=device)

        # 5. Generate
        with torch.no_grad():
            output = self.llm.generate(
                inputs_embeds=combined,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )

        return self.tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# Cell 6 - Recorder (fresh start)

from IPython.display import Javascript, display, Audio
from google.colab import output
import numpy as np
import io, base64, soundfile as sf
import torch
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

recorded_audio = {}  # global store

def recv_audio(data_url):
    header, encoded = data_url.split(",", 1)
    audio_bytes = base64.b64decode(encoded)
    buf = io.BytesIO(audio_bytes)
    try:
        audio_array, sr = sf.read(buf)
    except Exception:
        import subprocess
        with open("/tmp/input.webm", "wb") as f:
            f.write(audio_bytes)
        subprocess.run(
            ["ffmpeg", "-y", "-i", "/tmp/input.webm", "/tmp/input.wav"],
            capture_output=True
        )
        audio_array, sr = sf.read("/tmp/input.wav")

    if audio_array.ndim == 2:
        audio_array = audio_array.mean(axis=1)
    recorded_audio['array'] = audio_array.astype(np.float32)
    recorded_audio['sr']    = sr
    print(f"Got it! Duration: {len(audio_array)/sr:.1f}s | SR: {sr}Hz")
    print("Now run Cell 7")

output.register_callback('recv_audio', recv_audio)

display(Javascript("""
(function() {
  const old = document.getElementById('recorder-ui');
  if (old) old.remove();

  const ui = document.createElement('div');
  ui.id = 'recorder-ui';
  ui.style = 'font-family:sans-serif; padding:16px;';

  const statusEl = document.createElement('p');
  statusEl.id = 'rec-status';
  statusEl.style = 'font-size:16px; color:#555;';
  statusEl.innerText = 'Ready. Click Start to record.';

  const startBtn = document.createElement('button');
  startBtn.innerText = 'Start Recording';
  startBtn.style = 'padding:10px 24px; font-size:15px; background:#34a853; color:white; border:none; border-radius:6px; cursor:pointer; margin-right:12px;';

  const stopBtn = document.createElement('button');
  stopBtn.innerText = 'Stop & Send';
  stopBtn.style = 'padding:10px 24px; font-size:15px; background:#ea4335; color:white; border:none; border-radius:6px; cursor:pointer;';
  stopBtn.disabled = true;
  stopBtn.style.opacity = '0.5';

  ui.appendChild(statusEl);
  ui.appendChild(startBtn);
  ui.appendChild(stopBtn);
  document.body.appendChild(ui);

  let recorder, chunks = [];

  startBtn.onclick = async () => {
    chunks = [];
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    recorder = new MediaRecorder(stream);
    recorder.ondataavailable = e => { if (e.data.size > 0) chunks.push(e.data); };

    recorder.onstop = () => {
      stream.getTracks().forEach(t => t.stop());
      statusEl.innerText = 'Sending to Python...';
      statusEl.style.color = '#1a73e8';
      const blob = new Blob(chunks, { type: 'audio/webm' });
      const reader = new FileReader();
      reader.onloadend = () => {
        google.colab.kernel.invokeFunction('recv_audio', [reader.result], {});
        statusEl.innerText = 'Sent! Check output below.';
        statusEl.style.color = '#34a853';
        startBtn.disabled = false;
        startBtn.style.opacity = '1';
      };
      reader.readAsDataURL(blob);
    };

    recorder.start(100);
    statusEl.innerText = 'Recording... speak now!';
    statusEl.style.color = '#ea4335';
    startBtn.disabled = true;
    startBtn.style.opacity = '0.5';
    stopBtn.disabled = false;
    stopBtn.style.opacity = '1';
  };

  stopBtn.onclick = () => {
    if (recorder && recorder.state !== 'inactive') {
      recorder.stop();
      stopBtn.disabled = true;
      stopBtn.style.opacity = '0.5';
      statusEl.innerText = 'Processing...';
    }
  };
})();
"""))

Device: cuda


<IPython.core.display.Javascript object>

Got it! Duration: 5.2s | SR: 48000Hz
Now run Cell 7


In [ ]:
# Cell 7 - Complete Voice → Whisper → Qwen2.5 → Answer

import torch
import soundfile as sf
import numpy as np
import gc
from IPython.display import Audio, display
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Safety check
if 'recorded_audio' not in dir() or 'array' not in recorded_audio:
    print("No audio captured — run Cell 6 first and record your voice")
else:
    audio = recorded_audio['array']
    sr    = recorded_audio['sr']

    # ── Playback ──────────────────────────────────────────────────
    print("Playing back your recording...")
    display(Audio(audio, rate=sr))

    # ── Resample to 16kHz ─────────────────────────────────────────
    if sr != 16000:
        import torchaudio
        audio_tensor = torch.tensor(audio).unsqueeze(0)
        resampler = torchaudio.transforms.Resample(sr, 16000)
        audio = resampler(audio_tensor).squeeze().numpy()
        sr = 16000

    sf.write("/tmp/recorded.wav", audio, sr)

    # ── Step 1: Whisper on CPU ────────────────────────────────────
    print("\nStep 1: Transcribing your voice...")

    w_processor = WhisperProcessor.from_pretrained("openai/whisper-base")
    w_model = WhisperForConditionalGeneration.from_pretrained(
        "openai/whisper-base"
    )  # CPU intentionally — saves GPU for LLM

    inputs = w_processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        language="en",
        task="transcribe"
    ).input_features  # no .to(device) — stays on CPU

    with torch.no_grad():
        predicted_ids = w_model.generate(
            inputs,
            language="en",
            task="transcribe"
        )

    transcription = w_processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0].strip()

    print(f"You said: {transcription}")

    # Free Whisper immediately
    del w_model, w_processor, inputs, predicted_ids
    gc.collect()
    torch.cuda.empty_cache()
    print("Whisper freed from memory")

    # ── Step 2: Qwen2.5 1.5B in 4-bit ───────────────────────────
    print("\nStep 2: Loading Qwen2.5 and generating answer...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    qwen_tokenizer = AutoTokenizer.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct"
    )
    qwen_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct",
        quantization_config=bnb_config,
        device_map="auto"
    )

    messages = [
        {
            "role": "system",
            "content": "You are an accurate and helpful assistant. Answer questions with correct facts clearly and briefly."
        },
        {
            "role": "user",
            "content": transcription
        }
    ]

    prompt_text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    token_inputs = qwen_tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        output = qwen_model.generate(
            **token_inputs,
            max_new_tokens=200,
            do_sample=False,
            eos_token_id=qwen_tokenizer.eos_token_id
        )

    reply = qwen_tokenizer.decode(
        output[0][token_inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Clean up Qwen from memory
    del qwen_model, qwen_tokenizer, token_inputs, output
    gc.collect()
    torch.cuda.empty_cache()
    print("Qwen freed from memory")

    # ── Final Output ──────────────────────────────────────────────
    print("\n" + "="*50)
    print(f"You said  : {transcription}")
    print(f"ALM reply : {reply}")
    print("="*50)

Playing back your recording...



Step 1: Transcribing your voice...


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Qwen freed from memory

You said  : Hello, what is the captain of India?
ALM reply : The current captain of the Indian national cricket team (India A) is Hardik Pandya. He leads the team in domestic cricket matches like the Ranji Trophy and the One Day International series.
